# Quantize MXFP4 GGUF → Q4_K_M / IQ4_XS

**Key insight:** `save_pretrained_gguf(..., quantization_method='q4_k_m')` for GPT-OSS is intercepted by Unsloth — it produces a raw MXFP4 GGUF (~13.8G), NOT a Q4_K_M GGUF.  
That raw MXFP4 GGUF is the correct source to quantize.  
Running `llama-quantize mxfp4.gguf output.gguf Q4_K_M` on the compact MXFP4 source → ~11.6G (matches Unsloth original).  
Running `llama-quantize --imatrix ... IQ4_XS` → ~10.5G with better quality.

**Prerequisites (should already exist on this machine):**
- `gpt-oss-20b.MXFP4.gguf` — generated by previous notebook (Cell 1 will verify)
- `../llama.cpp/llama-quantize` — compiled llama.cpp binary
- `../llama.cpp/llama-imatrix` — for optional imatrix step

In [ ]:
# Cell 1: Check what exists
import os, subprocess

MXFP4_GGUF   = "gpt-oss-20b.MXFP4.gguf"
MXFP4_DIR    = "gpt-oss-20b-sft-mxfp4"     # fallback: merged HF dir
LLAMA_DIR    = "../llama.cpp"
QUANTIZER    = f"{LLAMA_DIR}/llama-quantize"
IMATRIX_BIN  = f"{LLAMA_DIR}/llama-imatrix"
CONVERTER    = f"{LLAMA_DIR}/convert_hf_to_gguf.py"

def file_size(path):
    if not os.path.exists(path): return "❌ not found"
    gb = os.path.getsize(path) / 1e9
    return f"{gb:.1f} GB"

print(f"{MXFP4_GGUF:<45} {file_size(MXFP4_GGUF)}")
print(f"{MXFP4_DIR:<45} {'✅ exists' if os.path.isdir(MXFP4_DIR) else '❌ not found'}")
print(f"{QUANTIZER:<45} {'✅ exists' if os.path.exists(QUANTIZER) else '❌ not found'}")
print(f"{IMATRIX_BIN:<45} {'✅ exists' if os.path.exists(IMATRIX_BIN) else '❌ not found'}")
print(f"{CONVERTER:<45} {'✅ exists' if os.path.exists(CONVERTER) else '❌ not found'}")

In [ ]:
# Cell 2: If MXFP4 GGUF is missing but the merged HF dir exists → convert it
# Skip this cell if gpt-oss-20b.MXFP4.gguf already exists.

if not os.path.exists(MXFP4_GGUF):
    if not os.path.isdir(MXFP4_DIR):
        raise RuntimeError(
            f"Neither {MXFP4_GGUF} nor {MXFP4_DIR} found.\n"
            "Re-run merge_and_export_gpt_oss_20b_MXFP4_Q4-K-M.ipynb cells 1–8 "
            "to regenerate the MXFP4 merged model first."
        )
    print(f"MXFP4 GGUF not found. Converting from {MXFP4_DIR} ...")
    cmd = [
        "python3", CONVERTER,
        MXFP4_DIR,
        "--outfile", MXFP4_GGUF,
    ]
    result = subprocess.run(cmd, check=True)
    print(f"Done. Size: {file_size(MXFP4_GGUF)}")
    # Expected: ~13.8 GB — if ~40 GB the converter dequantized to true BF16;
    # in that case use save_pretrained_gguf() path in Cell 2-alt below instead.
else:
    print(f"✅ Found {MXFP4_GGUF}: {file_size(MXFP4_GGUF)}")
    print("   Expected ≈ 13–14 GB (MXFP4 native). If ~40 GB, see Cell 2-alt.")

In [ ]:
# Cell 2-alt: Only run if Cell 2 shows MXFP4.gguf is ~40 GB (converter dequantized).
# Uses Unsloth's Python API directly — guaranteed to produce the compact MXFP4 GGUF.

# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained("unsloth/gpt-oss-20b")
# model = FastLanguageModel.get_peft_model(model, checkpoint_path="ospost/gpt-oss-20b-sft-adapter")
# model.save_pretrained_gguf("gpt-oss-20b-sft-out", tokenizer)
# # Unsloth intercepts GPT-OSS and saves as MXFP4 GGUF regardless of quantization_method
# # Output file will be gpt-oss-20b-sft-out.gguf (rename it to gpt-oss-20b.MXFP4.gguf)
# import os; os.rename("gpt-oss-20b-sft-out.gguf", MXFP4_GGUF)

print("Skipped — only needed if MXFP4.gguf is unexpectedly large (~40 GB).")

In [ ]:
# Cell 3: Quantize MXFP4 GGUF → Q4_K_M
# Expected output: ~11.6 GB (matches unsloth/gpt-oss-20b-GGUF/gpt-oss-20b-Q4_K_M.gguf)

OUTPUT_Q4 = "gpt-oss-20b-sft-Q4_K_M.gguf"

if os.path.exists(OUTPUT_Q4):
    print(f"Already exists: {OUTPUT_Q4} ({file_size(OUTPUT_Q4)}) — delete it to re-run.")
else:
    cmd = [QUANTIZER, MXFP4_GGUF, OUTPUT_Q4, "Q4_K_M"]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, check=True)
    print(f"\nDone. Output: {OUTPUT_Q4} — {file_size(OUTPUT_Q4)}")
    print("Target: ~11.6 GB")

In [ ]:
# Cell 4: Quick sanity test with llama-cli
# Uses --n-gpu-layers 99 to load entirely onto GPU.

TEST_PROMPT = "Explain step by step why the sky is blue."

cmd = [
    f"{LLAMA_DIR}/llama-cli",
    "--model", OUTPUT_Q4,
    "--prompt", TEST_PROMPT,
    "--n-gpu-layers", "99",
    "-n", "256",
    "--log-disable",
]
print("Running inference test ...\n")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-3000:] if result.stdout else "(no stdout)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## Optional: imatrix + IQ4_XS

IQ4_XS with importance-matrix calibration:
- **Size**: ~10.5 GB (vs 11.6 GB for Q4_K_M)
- **Quality**: better than Q4_K_M because critical attention weights are preserved at higher precision
- **Cost**: ~20 min extra for imatrix generation

Calibration data: use your own SFT training text for domain-specific importance scoring.

In [ ]:
# Cell 5: Prepare calibration data for imatrix
# Option A: export your SFT dataset (best for your domain)
# Option B: download wikitext (general purpose)

CALIB_FILE = "calibration.txt"

# --- Option A: export from your SFT dataset ---
# Uncomment and run if you prefer domain-specific calibration:
# from datasets import load_from_disk
# ds = load_from_disk("sft_dataset")   # adjust path
# with open(CALIB_FILE, "w") as f:
#     for row in ds.select(range(500)):
#         f.write(row["text"].strip() + "\n")
# print(f"Wrote {CALIB_FILE} from SFT dataset")

# --- Option B: wikitext-2 (default) ---
if not os.path.exists(CALIB_FILE):
    import urllib.request, zipfile
    url = "https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-2-raw-v1.zip"
    print(f"Downloading {url} ...")
    urllib.request.urlretrieve(url, "wikitext.zip")
    with zipfile.ZipFile("wikitext.zip") as z:
        z.extract("wikitext-2-raw/wiki.train.raw")
    os.rename("wikitext-2-raw/wiki.train.raw", CALIB_FILE)
    os.remove("wikitext.zip")
    print(f"✅ {CALIB_FILE} ready")
else:
    print(f"✅ {CALIB_FILE} already exists")

In [ ]:
# Cell 6: Generate imatrix (~20 min on A100/H100)
# Uses the MXFP4 GGUF (13.8G) as source model for importance scoring.

IMATRIX_FILE = "gpt-oss-20b-imatrix.dat"

if os.path.exists(IMATRIX_FILE):
    print(f"Already exists: {IMATRIX_FILE} — delete to regenerate.")
else:
    cmd = [
        IMATRIX_BIN,
        "-m", MXFP4_GGUF,
        "-f", CALIB_FILE,
        "-o", IMATRIX_FILE,
        "-ngl", "99",
        "--chunks", "128",
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, check=True)
    print(f"\n✅ {IMATRIX_FILE} ready")

In [ ]:
# Cell 7: Quantize with imatrix → IQ4_XS (~10.5 GB, better quality than Q4_K_M)

OUTPUT_IQ4 = "gpt-oss-20b-sft-IQ4_XS.gguf"

if os.path.exists(OUTPUT_IQ4):
    print(f"Already exists: {OUTPUT_IQ4} ({file_size(OUTPUT_IQ4)})")
else:
    cmd = [
        QUANTIZER,
        "--imatrix", IMATRIX_FILE,
        MXFP4_GGUF,
        OUTPUT_IQ4,
        "IQ4_XS",
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, check=True)
    print(f"\nDone. Output: {OUTPUT_IQ4} — {file_size(OUTPUT_IQ4)}")
    print("Target: ~10.5 GB")

In [ ]:
# Cell 8: Summary
files = {
    "MXFP4 GGUF (source)": MXFP4_GGUF,
    "Q4_K_M (target ~11.6G)": OUTPUT_Q4,
    "IQ4_XS + imatrix (target ~10.5G)": OUTPUT_IQ4,
}
print(f"{'File':<40} {'Size':>10}")
print("-" * 52)
for label, path in files.items():
    print(f"{label:<40} {file_size(path):>10}")